# 12. PySpark MLlib: Machine Learning Pipelines at Scale

When machine learning datasets exceed the RAM capacity of a single machine, traditional libraries like Scikit-Learn cannot train. **PySpark MLlib** provides distributed machine learning algorithms designed to scale across thousands of cores.

### Topics Covered:
1. **Introduction to MLlib: Transformers vs. Estimators**
2. **Feature Engineering: `StringIndexer`, `OneHotEncoder`, and `VectorAssembler`**
3. **Why `VectorAssembler` is mandatory in PySpark ML**
4. **Building an End-to-End Machine Learning `Pipeline`**
5. **Splitting Data: Training vs. Testing (`randomSplit`)**
6. **Training a Classification Model (`LogisticRegression`)**
7. **Evaluating Model Performance (`BinaryClassificationEvaluator` / ROC AUC)**
8. **Saving and Loading Trained Models**
9. **Stopping the SparkSession to free memory**


---
## 1. Initializing SparkSession


In [2]:


from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline, PipelineModel
import os
import shutil

spark = SparkSession.builder \
    .appName("12_MLlib_Pipelines") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession ready for Machine Learning!")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/11 19:02:22 WARN Utils: Your hostname, Tiger, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/11 19:02:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/mnt/d/Codes/AntiGravity/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/11 19:02:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession ready for Machine Learning!


---
## 2. Core Concepts: Transformers, Estimators, and Pipelines

* **Transformer**: An algorithm that converts one DataFrame into another by adding new feature columns (e.g. `VectorAssembler`, `OneHotEncoderModel`). Method: `.transform()`.
* **Estimator**: An algorithm that fits on a DataFrame to produce a Transformer (e.g. `LogisticRegression` fits to produce a `LogisticRegressionModel`). Method: `.fit()`.
* **Pipeline**: Chains multiple Transformers and Estimators together into a single cohesive, reproducible workflow. Eliminates data leakage!


---
## 3. Creating a Realistic Customer Churn Dataset

We want to predict whether a customer will **churn** (leave the service: 1 = Yes, 0 = No) based on their account features.


In [3]:
churn_data = [
    (1, "Female", 28, "Basic", 12.5, 45.0, 1),
    (2, "Male", 45, "Premium", 36.0, 95.0, 0),
    (3, "Male", 22, "Basic", 3.0, 30.0, 1),
    (4, "Female", 55, "Standard", 48.0, 70.0, 0),
    (5, "Female", 34, "Premium", 24.0, 85.0, 0),
    (6, "Male", 29, "Standard", 15.0, 60.0, 1),
    (7, "Female", 61, "Premium", 60.0, 110.0, 0),
    (8, "Male", 40, "Basic", 8.0, 40.0, 1),
    (9, "Female", 31, "Standard", 18.0, 65.0, 0),
    (10, "Male", 50, "Premium", 42.0, 100.0, 0),
    (11, "Female", 23, "Basic", 5.0, 35.0, 1),
    (12, "Male", 38, "Standard", 20.0, 68.0, 0)
]

columns = ["id", "gender", "age", "contract_type", "tenure_months", "monthly_charges", "churned"]

df = spark.createDataFrame(churn_data, columns)
print("Customer Churn Dataset:")
df.show()


Customer Churn Dataset:


+---+------+---+-------------+-------------+---------------+-------+
| id|gender|age|contract_type|tenure_months|monthly_charges|churned|
+---+------+---+-------------+-------------+---------------+-------+
|  1|Female| 28|        Basic|         12.5|           45.0|      1|
|  2|  Male| 45|      Premium|         36.0|           95.0|      0|
|  3|  Male| 22|        Basic|          3.0|           30.0|      1|
|  4|Female| 55|     Standard|         48.0|           70.0|      0|
|  5|Female| 34|      Premium|         24.0|           85.0|      0|
|  6|  Male| 29|     Standard|         15.0|           60.0|      1|
|  7|Female| 61|      Premium|         60.0|          110.0|      0|
|  8|  Male| 40|        Basic|          8.0|           40.0|      1|
|  9|Female| 31|     Standard|         18.0|           65.0|      0|
| 10|  Male| 50|      Premium|         42.0|          100.0|      0|
| 11|Female| 23|        Basic|          5.0|           35.0|      1|
| 12|  Male| 38|     Standard|    

---
## 4. Feature Engineering at Scale

Machine learning algorithms require numeric vector inputs:

1. **`StringIndexer`**: Encodes categorical strings into numerical indices (`"Basic"` $\rightarrow$ `0.0`, `"Standard"` $\rightarrow$ `1.0`, `"Premium"` $\rightarrow$ `2.0`).
2. **`OneHotEncoder`**: Converts category indices into binary one-hot vectors.
3. **`VectorAssembler`**: **(Mandatory)** PySpark models expect all features bundled into a single column named `features` containing a dense/sparse vector!


In [4]:
# Step 1: StringIndexer for categorical features
gender_indexer = StringIndexer(inputCol="gender", outputCol="gender_indexed")
contract_indexer = StringIndexer(inputCol="contract_type", outputCol="contract_indexed")

# Step 2: OneHotEncoder for the indexed categorical columns
encoder = OneHotEncoder(
    inputCols=["gender_indexed", "contract_indexed"],
    outputCols=["gender_vec", "contract_vec"]
)

# Step 3: VectorAssembler to combine numeric and encoded features into one vector
assembler = VectorAssembler(
    inputCols=["gender_vec", "contract_vec", "age", "tenure_months", "monthly_charges"],
    outputCol="features"
)


---
## 5. Building the Machine Learning `Pipeline`

Instead of manually fitting and transforming each step, we bundle everything into a `Pipeline`.


In [5]:
# Step 4: Define the Estimator (Classifier)
lr = LogisticRegression(featuresCol="features", labelCol="churned", maxIter=20)

# Step 5: Assemble the complete Pipeline
pipeline = Pipeline(stages=[
    gender_indexer,
    contract_indexer,
    encoder,
    assembler,
    lr
])

print("Pipeline configured with stages:")
for i, stage in enumerate(pipeline.getStages()):
    print(f"  Stage {i+1}: {stage.__class__.__name__}")


Pipeline configured with stages:
  Stage 1: StringIndexer
  Stage 2: StringIndexer
  Stage 3: OneHotEncoder
  Stage 4: VectorAssembler
  Stage 5: LogisticRegression


---
## 6. Train-Test Split & Model Training


In [6]:
# Split dataset: 80% Training, 20% Testing (set seed for reproducibility)
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows: {train_df.count()}, Testing rows: {test_df.count()}")

# Fit the entire pipeline on the training data with a single line!
pipeline_model = pipeline.fit(train_df)
print("Model training complete!")


Training rows: 11, Testing rows: 1


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


Model training complete!


---
## 7. Making Predictions & Evaluating Model Performance


In [7]:
# Transform test dataset to generate predictions
predictions = pipeline_model.transform(test_df)

# Predictions DataFrame contains rawPrediction, probability vector, and prediction (0.0 or 1.0)
predictions.select("id", "churned", "probability", "prediction").show(truncate=False)


+---+-------+------------------------------------------+----------+
|id |churned|probability                               |prediction|
+---+-------+------------------------------------------+----------+
|2  |0      |[0.9999999999999996,4.440892098500626E-16]|0.0       |
+---+-------+------------------------------------------+----------+



In [8]:
# Evaluate using Area Under ROC (BinaryClassificationEvaluator)
evaluator_roc = BinaryClassificationEvaluator(labelCol="churned", metricName="areaUnderROC")
roc_auc = evaluator_roc.evaluate(predictions)

# Evaluate Accuracy (MulticlassClassificationEvaluator)
evaluator_acc = MulticlassClassificationEvaluator(labelCol="churned", metricName="accuracy")
accuracy = evaluator_acc.evaluate(predictions)

print(f"Model Evaluation Results:")
print(f"  - ROC AUC: {roc_auc:.4f}")
print(f"  - Accuracy: {accuracy * 100:.2f}%")


Model Evaluation Results:
  - ROC AUC: 0.0000
  - Accuracy: 100.00%


---
## 8. Saving and Loading the Trained Model

In production, models are trained on cluster schedules and saved to storage (S3, ADLS, or local disk) for real-time serving.


In [9]:
model_dir = "saved_churn_model"

# Save the fitted pipeline model
pipeline_model.write().overwrite().save(model_dir)
print(f"Pipeline model successfully saved to: {model_dir}")

# Load the saved model back
loaded_model = PipelineModel.load(model_dir)
print("Pipeline model loaded successfully from disk!")

# Make a prediction with the loaded model on new incoming customer data
new_customer = spark.createDataFrame([
    (99, "Female", 25, "Basic", 2.0, 35.0, 0)
], columns)

loaded_predictions = loaded_model.transform(new_customer)
loaded_predictions.select("id", "probability", "prediction").show(truncate=False)

# Cleanup saved test model folder
if os.path.exists(model_dir):
    shutil.rmtree(model_dir)
    print("Cleaned up demo model directory.")


Pipeline model successfully saved to: saved_churn_model
Pipeline model loaded successfully from disk!
+---+---------------------------+----------+
|id |probability                |prediction|
+---+---------------------------+----------+
|99 |[5.350412599218514E-21,1.0]|1.0       |
+---+---------------------------+----------+

Cleaned up demo model directory.


---
## 9. Summary of Notebook 12
* PySpark MLlib trains machine learning models on distributed datasets.
* Use `StringIndexer` and `OneHotEncoder` for categorical feature encoding.
* Use `VectorAssembler` to merge features into a single `features` column.
* Build an end-to-end `Pipeline` containing both transformers and estimators.
* Evaluate models using `BinaryClassificationEvaluator` or `MulticlassClassificationEvaluator`.
* Save and reload models using `write().save()` and `PipelineModel.load()`.

---
## ⚠️ CRITICAL STEP: Stopping the SparkSession
Always release RAM and JVM background processes at the end of every notebook.


In [10]:
# Stop SparkSession to free memory
spark.stop()
print("SparkSession stopped successfully! RAM released.")


SparkSession stopped successfully! RAM released.
